# SPIKE-046: teacher runs in Colab

The runner (`jenab-scenarios`) and the teacher (`gemma4:31b` on Ollama) run in this VM. Each run is a real Jenab run: the real prompt, tools, SQLite database and asserts. Passing runs become training examples.

- GPU: a G4 or an A100, an L4 as the fallback. Cell 1 reads the card and picks the teacher's quantization, the context and the parallel runs from its memory.
- No keys: the repository is public, and a local Ollama needs none. `.env` never comes here.
- Results stay on the VM (`/content/spike046`) and cell 8 downloads them as one zip. A disconnect wipes the VM, so download before you disconnect; cell 8 also works halfway. Cell 5 downloads the held-out results by itself after each model.
- After a disconnect without Drive: run cells 1, 3 and the restore cell (upload the last zip), then go on. Cells 2 and 4 are checks and can be skipped; chunks in the zip are not run again.
- With `USE_DRIVE = True`, results go to Drive instead, and after a disconnect the cells resume where they stopped (finished chunks are skipped). Worth it for the long training run.
- Order: **1** settings and setup, **2** scripted smoke (no GPU), **3** Ollama and the models, **4** the 10-run check, **5** the held-out sets (the baseline and the ceiling), **6** the training sets, **7** the export, **8** the download.
- Run cell 5 before cell 6: if the base model already passes about 90% of the held-out tasks, there is nothing to train.
- Disconnect the runtime when you are done: units are spent while it is connected.

In [ ]:
# 1. Settings. Cell 1 reads the GPU and picks the teacher settings from its memory:
#   G4 (96 GB): gemma4:31b at 8-bit (q8_0, about 33 GB), 16k context, 4 runs at a time;
#               gpt-oss:120b as a second teacher if SECOND_TEACHER is True
#   A100 80 GB: gemma4:31b at 8-bit, 16k context, 4 runs at a time;
#               gpt-oss:120b as a second teacher if SECOND_TEACHER is True
#   A100 40 GB: gemma4:31b at 8-bit, 16k context, 1 run at a time
#   L4 (24 GB): gemma4:31b at 4-bit (q4_K_M), 8k context, 1 run at a time
# Set an OVERRIDE to change one setting; None keeps the pick. Then run every cell in order.
import os, subprocess

REPO = 'https://github.com/amir-saatchi/Jenab.git'
BRANCH = 'main'                # the branch with SPIKE-046's sets and export
GO_VERSION = '1.27.3'
REPS = 1                       # runs of each scenario
CHUNK = 10                     # scenarios per chunk; a chunk is saved to Drive when it ends
USE_DRIVE = False              # True: results in Drive, and the cells resume after a disconnect
SECOND_TEACHER = False         # 80 GB or more: gpt-oss:120b runs the training sets after gemma4
TEACHER_OVERRIDE = None        # such as 'gemma4:31b-it-q4_K_M'
CONTEXT_OVERRIDE = None        # such as 8192 if a 16k context runs out of memory
PARALLEL_OVERRIDE = None       # such as 2 if cell 3 or 5 says the teacher is partly in CPU memory
BASE_PARALLEL = 8              # runs at a time for the base model in cell 5; it is small
BASE = 'lfm3:1.7b'             # the model to train; its held-out score is the baseline
BASE_CONTEXT = 8192            # as on the PC (SPIKE-038)
HELDOUT_REPS = 3               # runs of each held-out scenario, for the base model and each teacher

smi = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits',
                     shell=True, capture_output=True, text=True)
GPU, GPU_MB = 'none', 0
if smi.returncode == 0 and smi.stdout.strip():
    GPU, mb = [x.strip() for x in smi.stdout.strip().splitlines()[0].rsplit(',', 1)]
    GPU_MB = int(mb)
print(f'GPU: {GPU}, {GPU_MB / 1024:.0f} GB')

if GPU_MB >= 90 * 1024:
    TEACHERS, CONTEXT, PARALLEL = ['gemma4:31b-it-q8_0'], 16384, 4
    if SECOND_TEACHER:
        TEACHERS.append('gpt-oss:120b')
elif GPU_MB >= 70 * 1024:
    TEACHERS, CONTEXT, PARALLEL = ['gemma4:31b-it-q8_0'], 16384, 4
    if SECOND_TEACHER:
        TEACHERS.append('gpt-oss:120b')
elif GPU_MB >= 35 * 1024:
    TEACHERS, CONTEXT, PARALLEL = ['gemma4:31b-it-q8_0'], 16384, 1
elif GPU_MB >= 20 * 1024:
    TEACHERS, CONTEXT, PARALLEL = ['gemma4:31b-it-q4_K_M'], 8192, 1
else:
    TEACHERS, CONTEXT, PARALLEL = [], 8192, 1
    print('No GPU with 20 GB or more: only cells 1 and 2 (the scripted smoke) can run.')
if SECOND_TEACHER and GPU_MB < 70 * 1024:
    print('SECOND_TEACHER needs 80 GB or more (an 80 GB A100 or a G4); it is off on this card.')
if TEACHER_OVERRIDE:
    TEACHERS[:1] = [TEACHER_OVERRIDE]
CONTEXT = CONTEXT_OVERRIDE or CONTEXT
PARALLEL = PARALLEL_OVERRIDE or PARALLEL
print(f'teachers: {TEACHERS}, context {CONTEXT}, parallel {PARALLEL}')

WORK = '/content/jenab'
OUT = '/content/drive/MyDrive/jenab-spike046' if USE_DRIVE else '/content/spike046'
os.environ.update({'PATH': '/usr/local/go/bin:' + os.environ['PATH'], 'CGO_ENABLED': '0'})

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)
print('output:', OUT)

In [ ]:
# 1. Go, the repository and the runner. CGO_ENABLED=0: the SQLite driver is pure Go.
# Also the set lists and the helpers that cells 4-6 use.
import subprocess, pathlib, glob, json, re, statistics, time

def sh(cmd, check=True):
    # Runs cmd and prints its output as it comes (Colab shows a child's output only through Python).
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    code = p.wait()
    if check and code != 0:
        raise SystemExit(f'failed ({code}): {cmd}')
    return code

if not pathlib.Path('/usr/local/go/bin/go').exists() or GO_VERSION not in subprocess.run('/usr/local/go/bin/go version', shell=True, capture_output=True, text=True).stdout:
    sh(f'curl -fsSL https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz -o /tmp/go.tgz')
    sh('rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz')
sh('go version')
if pathlib.Path(WORK).exists():
    sh(f'git -C {WORK} fetch --depth 1 origin {BRANCH} && git -C {WORK} checkout -q FETCH_HEAD')
else:
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO} {WORK}')
os.chdir(WORK)
sh('git log -1 --format="%h %s"')
sh('go build -o /content/jenab-scenarios ./cmd/jenab-scenarios')
JS = '/content/jenab-scenarios'
sh(f'{JS} check scenarios | grep sql-')

TRAIN = sorted(os.path.basename(d) for d in glob.glob('scenarios/sql-train-*'))
HELDOUT = sorted(os.path.basename(d) for d in glob.glob('scenarios/sql-heldout-*'))
print('training sets:', TRAIN)
print('held-out sets:', HELDOUT)

def scenario_ids(s):
    ids = []
    for p in sorted(glob.glob(f'scenarios/{s}/*.yaml')):
        if os.path.basename(p).startswith('_'):
            continue
        m = re.search(r'^id: (\S+)$', pathlib.Path(p).read_text(encoding='utf-8'), re.M)
        if m:
            ids.append(m.group(1))
    return ids

def newest_report(d):
    files = sorted(f for f in glob.glob(f'{d}/*.json') if re.search(r'/\d{8}-\d{6}\.json$', f))
    return json.load(open(files[-1], encoding='utf-8')) if files else None

def passed(r):
    return not r.get('error') and all(a['status'] != 'fail' or a.get('test') == 'info' for a in r['asserts'])

def times(rep):
    runs = [r for r in rep['runs'] if not r.get('skipped')]
    return runs, [r['seconds'] for r in runs], sum(map(passed, runs))

In [ ]:
# 2. Scripted smoke, no GPU and no model: every training set plays its _offline.yaml
# (the reference answers) through the real app, then the export runs on the reports.
# Checks the Linux build, the sets and the export before any GPU time is spent.
sh('rm -rf /content/smoke')
for s in TRAIN:
    sh(f'{JS} run -examples -parallel 4 -model scripted:scenarios/{s}/_offline.yaml -out /content/smoke/{s} scenarios/{s} | tail -1')
sh(f'{JS} export -out /content/smoke/export /content/smoke')
sh('wc -l /content/smoke/export/*.jsonl')

In [ ]:
# 3. Ollama, the teachers and the base model, as an ordinary ollama connection at 127.0.0.1:11434.
# The runner sends num_ctx from the context in models.yaml. One model is loaded at a time:
# with two teachers, the first runs every set, then the second.
import json, time, urllib.request

if not TEACHERS:
    raise SystemExit('no teacher fits this GPU (cell 1)')
if sh('which ollama', check=False) != 0:
    sh('which zstd || (apt-get update -qq && apt-get install -y -qq zstd)')  # the installer needs it
    sh('curl -fsSL https://ollama.com/install.sh | sh')

def ollama_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        return True
    except Exception:
        return False

SERVED = globals().get('SERVED')

def serve(parallel, context):
    # (Re)starts Ollama to serve `parallel` requests at a time. Its memory for a model is
    # about the weights plus context x parallel, so the number is set per model (cell 5).
    global SERVED
    if ollama_up() and SERVED == (parallel, context):
        return
    sh('pkill -x ollama', check=False)
    for _ in range(30):
        if not ollama_up():
            break
        time.sleep(1)
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(context), OLLAMA_KEEP_ALIVE='-1',
               OLLAMA_NUM_PARALLEL=str(parallel), OLLAMA_MAX_LOADED_MODELS='1')
    subprocess.Popen('ollama serve >> /content/ollama.log 2>&1', shell=True, env=env)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
    else:
        raise SystemExit('Ollama did not start; see /content/ollama.log')
    SERVED = (parallel, context)
    print(f'ollama: {parallel} at a time, context {context}', flush=True)

def load(m, context):
    # Loads m as the runner will use it and prints how much of it is on the GPU.
    t0 = time.time()
    req = urllib.request.Request('http://127.0.0.1:11434/api/generate', headers={'Content-Type': 'application/json'},
                                 data=json.dumps({'model': m, 'keep_alive': -1, 'options': {'num_ctx': context}}).encode())
    urllib.request.urlopen(req).read()
    for x in json.loads(urllib.request.urlopen('http://127.0.0.1:11434/api/ps').read())['models']:
        share = x['size_vram'] / x['size'] if x['size'] else 0
        print(f'{x["name"]}: {x["size"] / 1e9:.1f} GB with its context, {share:.0%} on the GPU, loaded in {time.time() - t0:.0f} s', flush=True)
        if share < 0.99:
            print('  part of it is in CPU memory, which is slow: lower the runs at a time (cell 1) or the context')

if not ollama_up() or SERVED is None:
    serve(PARALLEL, CONTEXT)

def pull(m):
    # Through the API, without streaming: `ollama pull` redraws its progress bar,
    # which a notebook prints as hundreds of blocks.
    print(f'pulling {m} ...', flush=True)
    t0 = time.time()
    req = urllib.request.Request('http://127.0.0.1:11434/api/pull', data=json.dumps({'model': m, 'stream': False}).encode(),
                                 headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req) as r:
        print(m, json.loads(r.read()).get('status'), f'in {time.time() - t0:.0f} s', flush=True)

for t in TEACHERS + [BASE]:
    pull(t)
sh('ollama list')

pathlib.Path('scenarios/models.yaml').write_text('# Colab: the teachers and the base model on the VM\'s own Ollama. No key.\n'
    'providers:\n  - name: colab\n    kind: ollama\n    base_url: http://127.0.0.1:11434/\n    models:\n'
    + ''.join(f'      - {{id: "{t}", context: {CONTEXT}}}\n' for t in TEACHERS)
    + f'      - {{id: "{BASE}", context: {BASE_CONTEXT}}}\n')
MODELS = [f'colab/{t}' for t in TEACHERS]
MODEL = MODELS[0]
BASE_MODEL = f'colab/{BASE}'
print('teachers:', MODELS, 'base:', BASE_MODEL)

In [ ]:
# 4. The 10-run check: 10 scenarios of the first training set on the first teacher,
# with the time per run.
# Use it to size the larger runs: runs per hour, and units per hour from Colab's resources panel.
# It can be skipped after a disconnect.
serve(PARALLEL, CONTEXT)
first = TRAIN[0]
ids = scenario_ids(first)[:10]
d = f'{OUT}/check10'
t0 = time.time()
sh(f'{JS} run -examples -model {MODEL} -parallel {PARALLEL} -out {d} -scenario {",".join(ids)} scenarios/{first}')
wall = time.time() - t0
runs, secs, ok = times(newest_report(d))
print(f'{len(runs)} runs, {ok} passed, wall {wall/60:.1f} min')
print(f'per run: mean {statistics.mean(secs):.0f} s, median {statistics.median(secs):.0f} s, max {max(secs):.0f} s')
print(f'about {3600 * len(runs) / wall:.0f} runs an hour at -parallel {PARALLEL}')
total = sum(len(scenario_ids(s)) for s in TRAIN) * REPS
print(f'the training sets: {total} runs a teacher, about {total * wall / len(runs) / 3600:.1f} h on {TEACHERS[0]}')

In [ ]:
# Restore (optional): after a disconnect without Drive, upload the last zip that cell 5 or
# cell 8 downloaded. It is unpacked into OUT, so cells 5 and 6 skip the chunks it holds
# (they have a DONE file) and run only the rest. Needs cells 1 and 3 first.
import zipfile
from google.colab import files

for name, data in files.upload().items():
    if os.path.exists(name):
        os.remove(name)  # upload() also saves it in the current folder, the repository
    z = f'/content/{os.path.basename(name)}'
    pathlib.Path(z).write_bytes(data)
    with zipfile.ZipFile(z) as f:
        names = f.namelist()
        if any(n.startswith('/') or '..' in n.split('/') for n in names):
            print(f'{name}: not a zip from this notebook, skipped')
            continue
        f.extractall(OUT)
    print(f'{name}: {len(names)} files into {OUT}')
for d in sorted(glob.glob(f'{OUT}/heldout/*/')) + sorted(glob.glob(f'{OUT}/reports/*/')):
    print(d, len(glob.glob(f'{d}*/*/DONE')), 'chunks done')

In [ ]:
# 5. The held-out sets: the base model (the baseline) and each teacher (the ceiling),
# HELDOUT_REPS reps each. Reports go to OUT/heldout, apart from the training reports, and
# without -examples: held-out runs are never training data (the export refuses them too).
# The chunks run side by side, one run each, as many as Ollama serves at a time:
# BASE_PARALLEL for the base model, PARALLEL for a teacher; Ollama restarts with that number
# before each model. A chunk with a DONE file is skipped, so the cell resumes, also after the
# restore cell. After each model the table is saved and heldout/ is downloaded as a zip.
# Pass rates here stand for the PC's; speed and RAM on the laptop are measured there.
import shutil, threading
from concurrent.futures import ThreadPoolExecutor

HELDOUT_CHUNK = 5          # scenarios per chunk
AUTO_DOWNLOAD = True       # download heldout/ after each model (not needed with Drive)

lock = threading.Lock()
procs = set()

def heldout_chunks(name):
    for s in HELDOUT:
        ids = scenario_ids(s)
        for i in range(0, len(ids), HELDOUT_CHUNK):
            part = ids[i:i + HELDOUT_CHUNK]
            yield s, f'{OUT}/heldout/{name.replace(":", "_")}/{s}/{i:02d}-{i + len(part) - 1:02d}', part

def heldout_chunk(name, model, s, d, ids):
    if not os.path.exists(f'{d}/DONE'):
        os.makedirs(d, exist_ok=True)
        t0 = time.time()
        with open(f'{d}/runner.log', 'w') as f:
            p = subprocess.Popen([JS, 'run', '-model', model, '-reps', str(HELDOUT_REPS), '-out', d,
                                  '-scenario', ','.join(ids), f'scenarios/{s}'], stdout=f, stderr=subprocess.STDOUT)
            with lock:
                procs.add(p)
            code = p.wait()
            with lock:
                procs.discard(p)
        rep = newest_report(d)
        if code != 0 or rep is None:
            tail = pathlib.Path(f'{d}/runner.log').read_text(errors='replace').splitlines()[-5:]
            print(f'{name} {s} {os.path.basename(d)}: the runner failed ({code})', *tail, sep='\n  ', flush=True)
            return None
        got, secs, ok = times(rep)
        line = f'{time.strftime("%Y-%m-%d %H:%M")} heldout {name} {s} {os.path.basename(d)}: {ok}/{len(got)} passed, {time.time() - t0:.0f} s'
        with lock:
            print(line, flush=True)
            with open(f'{OUT}/runs.log', 'a', encoding='utf-8') as log:
                log.write(line + '\n')
        pathlib.Path(f'{d}/DONE').write_text(line + '\n')
    return [dict(r, set=s) for r in newest_report(d)['runs'] if not r.get('skipped')]

def heldout_runs(name, model, slots, context):
    todo = list(heldout_chunks(name))
    left = sum(not os.path.exists(f'{d}/DONE') for _, d, _ in todo)
    print(f'{name}: {len(todo) - left} of {len(todo)} chunks done, {slots} runs at a time', flush=True)
    if left:
        serve(slots, context)
        load(name, context)
    ex = ThreadPoolExecutor(slots)
    try:
        results = list(ex.map(lambda c: heldout_chunk(name, model, *c), todo))
    except BaseException:  # such as a stop from the toolbar: stop the runners too
        ex.shutdown(wait=False, cancel_futures=True)
        with lock:
            for p in procs:
                p.kill()
        raise
    ex.shutdown()
    if None in results:
        print(f'{name}: a chunk failed; run the cell again to retry the chunks without DONE')
        return None
    return [r for rs in results for r in rs]

def save_heldout(tag):
    z = shutil.make_archive(f'/content/heldout-{tag}-{time.strftime("%Y%m%d-%H%M")}', 'zip', OUT, 'heldout')
    print(f'{z}: {os.path.getsize(z) / 1e6:.1f} MB', flush=True)
    if AUTO_DOWNLOAD and not USE_DRIVE:
        try:
            files.download(z)
        except Exception as e:
            print(f'the download failed ({e}); cell 8 downloads everything')

from google.colab import files
table = ['| Model | Passed | ' + ' | '.join(s.replace('sql-heldout-', '') for s in HELDOUT) + ' | Reps (least-most) | Median s a run |',
         '|---' * (len(HELDOUT) + 4) + '|']
share = lambda rs: sum(map(passed, rs)) / len(rs) if rs else 0
for name, model, slots, context in [(BASE, BASE_MODEL, BASE_PARALLEL, BASE_CONTEXT)] + [(t, m, PARALLEL, CONTEXT) for t, m in zip(TEACHERS, MODELS)]:
    t0 = time.time()
    runs = heldout_runs(name, model, slots, context)
    if runs is None:
        break
    per_set = [f'{share([r for r in runs if r["set"] == s]):.0%}' for s in HELDOUT]
    per_rep = [share([r for r in runs if r['rep'] == k]) for k in sorted({r['rep'] for r in runs})]
    table.append(f'| {name} | {sum(map(passed, runs))}/{len(runs)} ({share(runs):.0%}) | ' + ' | '.join(per_set)
                 + f' | {min(per_rep):.0%}-{max(per_rep):.0%} | {statistics.median(r["seconds"] for r in runs):.0f} |')
    summary = f'Held-out sets, {HELDOUT_REPS} reps, GPU {GPU}, {time.strftime("%Y-%m-%d")}\n\n' + '\n'.join(table) + '\n'
    pathlib.Path(f'{OUT}/heldout/summary.md').write_text(summary)
    print(f'{name} done in {(time.time() - t0) / 60:.0f} min\n\n{summary}', flush=True)
    save_heldout(name.replace(':', '_'))

In [ ]:
# 6. The training sets, in chunks of CHUNK scenarios, one teacher after the other. Each
# chunk writes its report and examples to Drive and a DONE file; a chunk with DONE is
# skipped, so after a disconnect this cell resumes where it stopped. A chunk cut halfway
# runs again from its start.
serve(PARALLEL, CONTEXT)
log = open(f'{OUT}/runs.log', 'a', encoding='utf-8')
stop = False
for t, model in zip(TEACHERS, MODELS):
    for s in TRAIN:
        ids = scenario_ids(s)
        for i in range(0, len(ids), CHUNK):
            d = f'{OUT}/reports/{t.replace(":", "_")}/{s}/chunk{i // CHUNK:02d}'
            if os.path.exists(f'{d}/DONE'):
                continue
            t0 = time.time()
            code = sh(f'{JS} run -examples -model {model} -reps {REPS} -parallel {PARALLEL} -out {d} -scenario {",".join(ids[i:i + CHUNK])} scenarios/{s} | tail -3', check=False)
            rep = newest_report(d)
            if code != 0 or rep is None:
                print(f'{t} {s} chunk {i // CHUNK}: the runner failed ({code}); run the cell again')
                stop = True
                break
            runs, secs, ok = times(rep)
            line = f'{time.strftime("%Y-%m-%d %H:%M")} {t} {s} chunk{i // CHUNK:02d}: {ok}/{len(runs)} passed, {time.time() - t0:.0f} s'
            print(line)
            log.write(line + '\n'); log.flush()
            pathlib.Path(f'{d}/DONE').write_text(line + '\n')
        if stop:
            break
    if stop:
        break
log.close()

In [ ]:
# 7. The export: every passing run of the training reports in Drive, as chat.jsonl and
# subagents.jsonl. It refuses held-out reports, and fails on personal IDs.
sh(f'{JS} export -out {OUT}/export {OUT}/reports')
sh(f'wc -l {OUT}/export/*.jsonl')
sh(f'tail -20 {OUT}/runs.log', check=False)

In [ ]:
# 8. The download: everything in OUT as one zip (reports, held-out table, export, runs.log).
# Run it whenever you want a copy, and always before you disconnect without Drive.
import os, shutil, time
from google.colab import files

z = shutil.make_archive(f'/content/jenab-spike046-{time.strftime("%Y%m%d-%H%M")}', 'zip', OUT)
print(f'{z}: {os.path.getsize(z) / 1e6:.1f} MB')
files.download(z)

## After the runs

- Note in the ticket: the held-out table (`heldout/summary.md` in Drive), the training runs, the passes, the failures and why (the reports' Markdown files say), the GPU hours and the units.
- Download first (cell 8) unless the results are in Drive. Then disconnect: **Runtime → Disconnect and delete runtime**.